In [16]:
! pip install sentence-transformers faiss-cpu numpy

In [17]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
def chunk_text(text, chunk_size=30, overlap=5):

    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [19]:
text = """
Artificial Intelligence (AI) is a field that focuses on creating machines capable of intelligent behavior.
Machine Learning (ML) is a subset of AI that enables machines to learn from data.
Deep Learning (DL) uses neural networks with multiple layers for complex pattern recognition.
Applications of AI include natural language processing, computer vision, robotics, healthcare, finance, and recommendation systems.
Ethical AI ensures fairness, accountability, and transparency in AI deployments.
AI continues to evolve rapidly, impacting numerous industries and research domains.
"""

In [20]:
chunks = chunk_text(text)

for i, c in enumerate(chunks):
    print(f"Chunk {i}:" , c)

Chunk 0: Artificial Intelligence (AI) is a field that focuses on creating machines capable of intelligent behavior. Machine Learning (ML) is a subset of AI that enables machines to learn from data.
Chunk 1: machines to learn from data. Deep Learning (DL) uses neural networks with multiple layers for complex pattern recognition. Applications of AI include natural language processing, computer vision, robotics, healthcare, finance,
Chunk 2: computer vision, robotics, healthcare, finance, and recommendation systems. Ethical AI ensures fairness, accountability, and transparency in AI deployments. AI continues to evolve rapidly, impacting numerous industries and research domains.
Chunk 3: industries and research domains.


In [21]:
chunk_embeddings = model.encode(chunks)

In [22]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(chunk_embeddings))

In [23]:
question = input("Enter your question: ")

Enter your question: what is ML


In [24]:
question_embedding = model.encode([question])

In [25]:
k = 2

distances, indices = index.search(np.array(question_embedding), k)

retrieved_chunks = [chunks[i] for i in indices[0]]

print("Retrieved Chunks:")
for c in retrieved_chunks:
    print("-", c)

Retrieved Chunks:
- Artificial Intelligence (AI) is a field that focuses on creating machines capable of intelligent behavior. Machine Learning (ML) is a subset of AI that enables machines to learn from data.
- machines to learn from data. Deep Learning (DL) uses neural networks with multiple layers for complex pattern recognition. Applications of AI include natural language processing, computer vision, robotics, healthcare, finance,


In [30]:
context = " ".join(retrieved_chunks)

answer = f"""
Based on the document:
{question}

Answer:
{context}
"""

print(answer)


Based on the document:
what is ML

Answer:
Artificial Intelligence (AI) is a field that focuses on creating machines capable of intelligent behavior. Machine Learning (ML) is a subset of AI that enables machines to learn from data. machines to learn from data. Deep Learning (DL) uses neural networks with multiple layers for complex pattern recognition. Applications of AI include natural language processing, computer vision, robotics, healthcare, finance,



In [31]:
def summarize_document(text, max_words=100):

    words = text.split()

    if len(words) > max_words:
        summary = " ".join(words[:max_words]) + "..."
        return summary

    return text